# Stage 9 — Debit spread ranking

Full-universe backtest (notebook 14, 214 instruments) showed 59.3% with a
positive avg R but high variance across instruments (std 0.21 vs mean 0.08) -
not a uniform edge. This ranks instruments the same way the other three
strategies are ranked (`strategy_scoring.py`), so the shortlist is the
instruments where this specific signal+lifecycle actually worked, not the
whole universe.

median_r_achieved is weighted highest (not avg_r_achieved) - same
median-over-mean preference as the rest of this project's stats, and it's
what caught BHARTIARTL's outlier-reliant result during validation.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.strategy_scoring import score_for_debit_spread, save_ranking

comparison = pd.read_excel("../exports/DEBIT_SPREAD_COMPARISON.xlsx", sheet_name="Comparison")
comparison.shape

(224, 7)

In [2]:
ranked, watchlist = score_for_debit_spread(comparison)
ranked.head(20)

,symbol,debit_spread_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,POWERINDIA,98.6,,69,62.3,0.49,1.50,5.0
1,NIFTY,94.8,,64,53.1,0.31,0.64,5.0
2,SWIGGY,94.5,,31,64.5,0.52,1.50,6.0
3,CDSL,94.1,,71,57.7,0.42,1.50,6.0
4,PFC,93.2,,65,55.4,0.37,0.95,6.0
5,TIINDIA,92.8,,68,52.9,0.27,0.37,5.0
6,INFY,92.3,,76,55.3,0.32,0.74,6.0
7,BDL,91.5,,76,52.6,0.24,0.32,5.0
8,RELIANCE,91.4,,78,53.8,0.32,0.79,6.0
9,BHEL,91.3,,60,56.7,0.32,0.45,6.0


In [3]:
# How many top-20 candidates carry the outlier-reliance warning?
print(f"Instruments passing min_trades filter: {len(ranked)} of {len(comparison)}")
print(f"Flagged in top 20: {(ranked.head(20)['outlier_reliance_flag'] != '').sum()}")
ranked[ranked["outlier_reliance_flag"] != ""].head(10)

Instruments passing min_trades filter: 222 of 224
Flagged in top 20: 0


,symbol,debit_spread_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
36,COFORGE,79.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.32R),75,49.3,0.22,-0.10,5.0
37,SAIL,78.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.32R),64,50.0,0.22,-0.10,5.5
46,GODREJPROP,74.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.30R),64,50.0,0.20,-0.10,6.0
55,MIDCPNIFTY,69.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.62R),88,48.9,0.18,-0.44,5.0
56,DABUR,69.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),61,49.2,0.16,-0.20,6.0
57,BANDHANBNK,69.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.37R),76,48.7,0.17,-0.20,6.0
58,ONGC,68.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.31R),67,46.3,0.21,-0.10,6.0
59,JSWENERGY,68.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.70R),67,49.3,0.17,-0.53,5.0
60,SBIN,68.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.51R),65,49.2,0.19,-0.32,6.0
63,ASHOKLEY,66.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.41R),73,47.9,0.16,-0.25,6.0


In [4]:
save_ranking(ranked, "RANKING_debit_spread.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/RANKING_debit_spread.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/RANKING_debit_spread.xlsx'

In [5]:
# Instruments below min_trades - too thin a sample to blend into the
# percentile ranking above, but not dropped either (e.g. SWIGGY: recently
# added to F&O, so it hasn't had time to generate enough signals yet).
# Sorted by avg_r_achieved so a promising-but-unproven name is visible,
# not silently missing. Re-check at the next periodic refresh - trade
# count grows on its own as more calendar time passes.
print(f"Instruments on watchlist (below min_trades): {len(watchlist)}")
watchlist

Instruments on watchlist (below min_trades): 2


,symbol,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,ATHERENERG,29,62.1,0.46,1.5,5.0
1,ENRIN,24,37.5,-0.11,-1.0,6.0


In [6]:
save_ranking(watchlist, "WATCHLIST_debit_spread.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_debit_spread.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_debit_spread.xlsx'